<style>
:root { --jp-content-font-size1: 11px; --jp-code-font-size: 10px; --jp-ui-font-size1: 10px; }
body { font-size: 11px; line-height: 1.4; }
h1 { font-size: 20px !important; }
h2 { font-size: 16px !important; margin-top: 1.2em !important; }
h3 { font-size: 13px !important; }
table { font-size: 9px !important; table-layout: auto !important; }
th, td { font-size: 9px !important; padding: 2px 4px !important; }
.jp-Cell { page-break-inside: avoid; }
.page-break { page-break-before: always; }
pre { font-size: 9px !important; line-height: 1.3 !important; }
@page { margin: 0.6in 0.5in; }
</style>

# Local Embeddings with FastEmbed

This notebook compares multiple **local embedding models** from FastEmbed
(ONNX-based, no GPU required) for a document retrieval task.

We embed the same set of documents and a query with three different models,
then compare how each model ranks the documents by similarity.

No API keys needed -- all models run locally.

<div class="page-break"></div>

---

## Section 1: Setup & Model Selection

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="coroutine.*was never awaited", category=RuntimeWarning)

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / "pyproject.toml").is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

from fastembed import TextEmbedding  # noqa: E402

from ffai.rag.embed import Embeddings  # noqa: E402

print(f"Project root: {_project_root}")
print("Imports successful")


In [ ]:
MODELS = [
    ('BAAI/bge-small-en-v1.5', 384),
    ('BAAI/bge-base-en-v1.5', 768),
    ('sentence-transformers/all-MiniLM-L6-v2', 384),
]

models = {}
for model_name, expected_dim in MODELS:
    models[model_name] = TextEmbedding(model_name)
    print(f'Loaded: {model_name} (expected dim={expected_dim})')

print(f'\n{len(models)} models loaded')

<div class="page-break"></div>

---

## Section 2: Documents & Query

In [ ]:
documents = [
    'Kubernetes automates deployment, scaling, and management of containerized applications.',
    'Docker containers package applications with their dependencies for consistent execution.',
    'The French Revolution began in 1789 and led to major political changes in Europe.',
    'Terraform is an infrastructure-as-code tool that manages cloud resources declaratively.',
    'Photosynthesis converts sunlight, water, and CO2 into glucose and oxygen in plants.',
    'CI/CD pipelines automate building, testing, and deploying software changes.',
    'The periodic table organizes chemical elements by atomic number and properties.',
    'Prometheus monitors system metrics and triggers alerts based on threshold rules.',
]

query = 'How do teams automate software deployment and infrastructure?'

print(f'Documents: {len(documents)}')
print(f'Query: "{query}"')

<div class="page-break"></div>

---

## Section 3: Embed with Each Model

Each model produces embeddings with different dimensions and semantic properties.
We embed all documents and the query with every model, then compare rankings.

In [ ]:
import numpy as np


def embed_texts(model, texts):
    return np.array(list(model.embed(texts)))


def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


all_results = {}

for model_name, model in models.items():
    doc_vecs = embed_texts(model, documents)
    query_vec = embed_texts(model, [query])[0]

    scores = [(i, cosine_sim(query_vec, dv)) for i, dv in enumerate(doc_vecs)]
    scores.sort(key=lambda x: x[1], reverse=True)

    all_results[model_name] = {
        'doc_vecs': doc_vecs,
        'query_vec': query_vec,
        'rankings': scores,
    }

    print(f'{model_name}:')
    print(f'  Embedding dim: {doc_vecs.shape[1]}')
    print(f'  Top 3 docs: {[(i, round(s, 4)) for i, s in scores[:3]]}')
    print()

<div class="page-break"></div>

---

## Section 4: Side-by-Side Ranking Comparison

Compare how each model ranks the same documents for the same query.
Agreement across models indicates robust relevance.

In [ ]:
import polars as pl

rows = []
for model_name, data in all_results.items():
    for rank, (doc_idx, score) in enumerate(data['rankings'], 1):
        rows.append({
            'model': model_name.split('/')[-1],
            'rank': rank,
            'doc_idx': doc_idx,
            'score': round(score, 4),
            'text': documents[doc_idx][:60],
        })

df = pl.DataFrame(rows)
print(df)

<div class="page-break"></div>

---

## Section 5: Cross-Model Agreement

Which documents consistently rank in the top 3 across all models?
These are the most robustly relevant results.

In [ ]:
top3_per_model = {}
for model_name, data in all_results.items():
    top3 = {doc_idx for doc_idx, _ in data['rankings'][:3]}
    top3_per_model[model_name] = top3

all_top3 = set.intersection(*top3_per_model.values())
any_top3 = set.union(*top3_per_model.values())

print('Documents in top-3 for ALL models (consistently relevant):')
for doc_idx in sorted(all_top3):
    print(f'  Doc {doc_idx}: {documents[doc_idx][:80]}')

print()
print('Documents in top-3 for ANY model:')
for doc_idx in sorted(any_top3):
    in_all = 'ALL' if doc_idx in all_top3 else 'some'
    print(f'  Doc {doc_idx} ({in_all}): {documents[doc_idx][:80]}')

<div class="page-break"></div>

---

## Section 6: Embedding Dimension Comparison

Smaller dimensions mean faster similarity search but potentially less
semantic nuance. Larger dimensions capture more detail at higher cost.

In [ ]:
dim_rows = []
for model_name, data in all_results.items():
    dim = data['doc_vecs'].shape[1]
    top1_score = data['rankings'][0][1]
    dim_rows.append({
        'model': model_name.split('/')[-1],
        'dimension': dim,
        'top1_score': round(top1_score, 4),
        'embedding_size_kb': round(dim * 4 / 1024, 1),
    })

dim_df = pl.DataFrame(dim_rows)
print(dim_df)

<div class="page-break"></div>

---

## Section 7: FastEmbed vs API Embeddings Comparison

Compare the local FastEmbed rankings against the Mistral API embeddings.
This shows whether a free local model can match a paid API for retrieval.

In [ ]:
import os

api_key = os.getenv('MISTRAL_API_KEY')

if api_key:
    api_emb = Embeddings(model='mistral/mistral-embed', api_key=api_key)
    api_doc_vecs = api_emb.embed(documents)
    api_query_vec = api_emb.embed_single(query)

    api_scores = [
        (i, Embeddings.cosine_similarity(api_query_vec, dv))
        for i, dv in enumerate(api_doc_vecs)
    ]
    api_scores.sort(key=lambda x: x[1], reverse=True)

    print('Mistral API (mistral-embed) rankings:')
    for rank, (doc_idx, score) in enumerate(api_scores[:5], 1):
        print(f'  {rank}. [score={score:.4f}] {documents[doc_idx][:60]}')

    api_top3 = {idx for idx, _ in api_scores[:3]}
    print(f'\nMistral API top-3 doc indices: {sorted(api_top3)}')
    print(f'FastEmbed consensus top-3:      {sorted(all_top3)}')
    print(f'Overlap: {sorted(api_top3 & all_top3)}')
else:
    print('MISTRAL_API_KEY not set -- skipping API comparison')
    print('Set MISTRAL_API_KEY in .env to compare local vs API embeddings')

<div class="page-break"></div>

---

## Section 8: Using FastEmbed with FFAI's Local Embeddings

FFAI's `Embeddings` class supports local models via the `local/` prefix.
This uses sentence-transformers under the hood but works seamlessly
with FastEmbed-compatible model names for local inference.

In [ ]:
local_models = [
    'local/all-MiniLM-L6-v2',
    'local/BAAI/bge-small-en-v1.5',
]

for model_str in local_models:
    try:
        emb = Embeddings(model=model_str)
        vec = emb.embed_single('test embedding')
        print(f'{model_str:40s}  dim={len(vec):>4}  is_local={emb.is_local}')
    except Exception as e:
        print(f'{model_str:40s}  ERROR: {e}')

---

## Key Takeaways

| Topic | Insight |
|-------|---------|
| **FastEmbed models** | ONNX-based, no GPU, ~100MB total vs multi-GB for sentence-transformers |
| **Dimension tradeoff** | Smaller dims (384) are faster; larger dims (768-1024) capture more nuance |
| **Cross-model agreement** | Documents ranking high across multiple models are most robustly relevant |
| **Local vs API** | Local models are free and fast; API models may better capture domain-specific semantics |
| **Use case** | FastEmbed is ideal for prototyping, offline search, and cost-sensitive production |